# Start

In [1]:
# import libraries

# import pandas for manipulation
import pandas as pd

# mount google drive
from google.colab import drive
drive.mount('/content/drive')
from google.colab import userdata

# Library needed to connect to the database
from sqlalchemy import create_engine, text
import psycopg2

!pip install xlsxwriter

MessageError: Error: credential propagation was unsuccessful

In [ ]:
# name export path
export_gen_path = "/content/drive/Shareddrives/GEM Shared Drive/Programs/Oil & Gas Program/Global Oil and Gas Extraction Tracker (GOGET)/GOGET Iterations/Alternative Formats of GOGET/"
export_qc_path = '/content/drive/Shareddrives/GEM Shared Drive/Programs/Oil & Gas Program/Global Oil and Gas Extraction Tracker (GOGET)/QC/QC Output Dump/'

In [ ]:
# make timestamp
from datetime import datetime
now = datetime.now()
timestamp = now.strftime("%Y-%m-%d_%H-%M-%S")

print('this runs qc output will all be timestamped: ' + timestamp)

this runs qc output will all be timestamped: 2026-04-06_13-48-13


This is the url we use to connect to the GEM database. This one specific connects to a read-only database that David set up so that people can query the database without worry about editing the database accidentally.

Run the following to cells to connect this jupyter notebook to the database. This will allow you to run queries from this notebook.

In [ ]:
# Set up database pull via URL from the data team.
# (URL updated 2026-02-17 by Scott to reflect new follower database location)
DATABASE_URL = userdata.get('DATABASE_URL')

if DATABASE_URL.startswith('postgres:'):
    DATABASE_URL = DATABASE_URL.replace('postgres:', 'postgresql:')

# Connect to the database
engine = create_engine(DATABASE_URL)
conn = engine.connect()

In [ ]:
# create avaialable table list
def available_tables():
  # Show names of all tables
    query = text("""
        SELECT table_name
        FROM information_schema.tables
        WHERE table_schema = 'public'
        ORDER BY table_name
        """)
    table_names = pd.read_sql(query, conn)
    table_names = table_names['table_name'].tolist()
    return table_names

available_tables()

['account_emailaddress',
 'account_emailconfirmation',
 'auth_group',
 'auth_group_permissions',
 'auth_permission',
 'auth_user',
 'auth_user_groups',
 'auth_user_user_permissions',
 'capacity_rating',
 'captive_industry_type',
 'captive_industry_use',
 'captive_non_industry_use',
 'ccs',
 'chp',
 'comment',
 'company',
 'company_owner',
 'country',
 'country_subdivision',
 'data_source',
 'django_admin_log',
 'django_content_type',
 'django_migrations',
 'django_session',
 'django_site',
 'entity_history',
 'entity_org_id',
 'entity_status',
 'entity_tag',
 'entity_type',
 'entity_update',
 'external_id_system',
 'fuel_category',
 'fuel_detail',
 'goget_project',
 'hydrogen_capable',
 'hydrogen_generating',
 'hydrogen_greenwashing',
 'installation_type',
 'iron_yearly_production',
 'language',
 'legal_entity_type',
 'lng_project',
 'lng_unit',
 'nuclear_model',
 'operator',
 'org_id',
 'plant',
 'plant_external_id',
 'plant_history',
 'plant_language',
 'plant_owner',
 'powerplant_un

In [ ]:
# function to convert sql tables to dataframes
def get_sql_table(table_name):
   df = pd.read_sql(table_name,conn)
   return df

In [ ]:
plant = get_sql_table("plant")

In [ ]:
country = get_sql_table("country")
geo = get_sql_table("project_geospatial")
goget_project = get_sql_table("goget_project")

# print loaded
print("loaded plant, country, geo, and goget")

loaded plant, country, geo, and goget


In [ ]:
# rename ID in plant as "plant_table_id"
plant = plant.rename(columns={'id': 'plant_table_id'})

# Ensure 'plant_table_id' is of string type for merging
plant['plant_table_id'] = plant['plant_table_id'].astype(str)

# Ensure 'project_id' in goget_project is also of string type for merging
goget_project['project_id'] = goget_project['project_id'].astype(str)

# merge country from plant id
goget_project = goget_project.merge(plant[['plant_table_id', 'country_id', 'latitude', 'longitude' ]], left_on='project_id', right_on='plant_table_id', how='left')

display(goget_project.head())

,id,basinDatasource,concessionBlockDatasource,projectComplexDatasource,offshore,offshoreDatasource,productionType,productionTypeDatasource,fuelType,unitType,...,concessionBlock4,concessionBlock5,concessionBlock6,concessionBlock7,capex,capexDatasource,plant_table_id,country_id,latitude,longitude
0,11409,[],[],[],onshore,[193518],None,[],oil,pool,...,None,None,None,None,NaN,[],100000313656,41.0,51.9190,-109.2960
1,11782,[],[],[],offshore,[193519],None,[],oil and gas,asset,...,None,None,None,None,NaN,[],100000314273,239.0,28.9310,-90.3340
2,11783,[],[],[],offshore,[193519],None,[],oil and gas,asset,...,None,None,None,None,NaN,[],100000314274,239.0,28.4320,-91.4735
3,11785,[],[],[],offshore,[193519],None,[],oil and gas,asset,...,None,None,None,None,NaN,[],100000314276,239.0,27.9990,-93.9600
4,11787,[],[],[],offshore,[193519],None,[],oil and gas,asset,...,None,None,None,None,NaN,[],100000314278,239.0,28.5463,-91.5304


In [ ]:
# make ID a string, dropping tailing 0
goget_project['country_id'] = pd.to_numeric(goget_project['country_id'], errors='coerce')
goget_project['country_id'] = goget_project['country_id'].fillna(-1).astype('int64')

# merge country onto country ID from plant
goget_project = goget_project.merge(country[['id', 'gemName']], left_on='country_id', right_on='id', how='left')

In [ ]:
goget_project = goget_project.rename(columns={'gemName': 'Country/Area'})
goget_project

,id_x,basinDatasource,concessionBlockDatasource,projectComplexDatasource,offshore,offshoreDatasource,productionType,productionTypeDatasource,fuelType,unitType,...,concessionBlock6,concessionBlock7,capex,capexDatasource,plant_table_id,country_id,latitude,longitude,id_y,Country/Area
0,11409,[],[],[],onshore,[193518],None,[],oil,pool,...,None,None,NaN,[],100000313656,41,51.9190,-109.2960,41.0,Canada
1,11782,[],[],[],offshore,[193519],None,[],oil and gas,asset,...,None,None,NaN,[],100000314273,239,28.9310,-90.3340,239.0,United States
2,11783,[],[],[],offshore,[193519],None,[],oil and gas,asset,...,None,None,NaN,[],100000314274,239,28.4320,-91.4735,239.0,United States
3,11785,[],[],[],offshore,[193519],None,[],oil and gas,asset,...,None,None,NaN,[],100000314276,239,27.9990,-93.9600,239.0,United States
4,11787,[],[],[],offshore,[193519],None,[],oil and gas,asset,...,None,None,NaN,[],100000314278,239,28.5463,-91.5304,239.0,United States
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
17222,11372,[],[],[],onshore,[193518],None,[],oil,pool,...,None,None,NaN,[],100000313619,41,50.1060,-103.4450,41.0,Canada
17223,11233,[],[],[],onshore,[193518],None,[],oil,pool,...,None,None,NaN,[],100000313479,41,49.7090,-103.2110,41.0,Canada
17224,8370,[],[],[193249],onshore,[193518],unconventional,[193249],oil and gas,asset,...,None,None,NaN,[],100000309565,41,53.7080,-110.2530,41.0,Canada
17225,13220,[],[],[],onshore,[193518],None,[],oil and gas,asset,...,None,None,NaN,[],100000319245,239,34.6073,-96.1555,239.0,United States


In [ ]:
# add wkt to goget_project using project_id from goget_project and project_id from geo as key column
geo['project_id'] = geo['project_id'].astype(str)
goget_project_wkt = goget_project.merge(geo[['project_id', 'wkt']], left_on='project_id', right_on='project_id', how='left')

# check project id = 100001081837 for sucess
display(goget_project_wkt[goget_project_wkt['project_id'] == '100001081837'])

# save out goget_project_wkt
goget_project_wkt.to_excel(export_gen_path + 'goget_project_wkt_' + timestamp + '.xlsx', index=False)

,id_x,basinDatasource,concessionBlockDatasource,projectComplexDatasource,offshore,offshoreDatasource,productionType,productionTypeDatasource,fuelType,unitType,...,concessionBlock7,capex,capexDatasource,plant_table_id,country_id,latitude,longitude,id_y,gemName,wkt
6640,16753,[],[188770],[],offshore,[193519],conventional,[],gas,field,...,None,NaN,[],100001081837,7,-6.4551,12.0862,7.0,Angola,NaN


In [ ]:
# drop country_id	and	id_y
goget_project = goget_project.drop(columns=['country_id', 'id_y'])


In [ ]:
goget_project

,id_x,basinDatasource,concessionBlockDatasource,projectComplexDatasource,offshore,offshoreDatasource,productionType,productionTypeDatasource,fuelType,unitType,...,concessionBlock4,concessionBlock5,concessionBlock6,concessionBlock7,capex,capexDatasource,plant_table_id,latitude,longitude,gemName
0,11409,[],[],[],onshore,[193518],None,[],oil,pool,...,None,None,None,None,NaN,[],100000313656,51.9190,-109.2960,Canada
1,11782,[],[],[],offshore,[193519],None,[],oil and gas,asset,...,None,None,None,None,NaN,[],100000314273,28.9310,-90.3340,United States
2,11783,[],[],[],offshore,[193519],None,[],oil and gas,asset,...,None,None,None,None,NaN,[],100000314274,28.4320,-91.4735,United States
3,11785,[],[],[],offshore,[193519],None,[],oil and gas,asset,...,None,None,None,None,NaN,[],100000314276,27.9990,-93.9600,United States
4,11787,[],[],[],offshore,[193519],None,[],oil and gas,asset,...,None,None,None,None,NaN,[],100000314278,28.5463,-91.5304,United States
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
17268,11372,[],[],[],onshore,[193518],None,[],oil,pool,...,None,None,None,None,NaN,[],100000313619,50.1060,-103.4450,Canada
17269,11233,[],[],[],onshore,[193518],None,[],oil,pool,...,None,None,None,None,NaN,[],100000313479,49.7090,-103.2110,Canada
17270,8370,[],[],[193249],onshore,[193518],unconventional,[193249],oil and gas,asset,...,None,None,None,None,NaN,[],100000309565,53.7080,-110.2530,Canada
17271,13220,[],[],[],onshore,[193518],None,[],oil and gas,asset,...,None,None,None,None,NaN,[],100000319245,34.6073,-96.1555,United States


In [ ]:
# filter status_timeline for only entries in goget_project
status_timeline = get_sql_table("status_timeline")

# Convert 'unit_id' in status_timeline to string type to match 'project_id' in goget_project
status_timeline['unit_id'] = status_timeline['unit_id'].astype(str)

# Perform an inner merge using unit_id from status_timeline and project_id from goget_project
status_timeline_filtered = status_timeline.merge(goget_project[['project_id']], left_on='unit_id', right_on='project_id', how='inner')

# Select only the original columns from status_timeline
status_timeline_merged = status_timeline_filtered[status_timeline.columns]

display(status_timeline_merged.head())

,id,order,status,substatus,statusDatasource,year,monthOrHalfYear,unit_id,created,modified,delayed,notes
0,19876,1,discovered,,[193245],1978.0,None,100000309032,2025-09-10 16:46:04.994321+00:00,2026-02-27 12:43:12.552439+00:00,False,None
1,17136,2,operating,ramp up,[193191],1995.0,None,100000301072,2025-09-10 16:45:45.326477+00:00,2026-02-27 12:30:06.169041+00:00,False,None
2,24670,3,operating,,[212075],2025.0,,100000312250,2025-09-10 16:46:37.662886+00:00,2026-02-27 12:43:43.871888+00:00,False,
3,8352,1,discovered,,[193400],1963.0,None,100000302911,2025-08-06 21:45:09.234914+00:00,2026-02-27 12:44:28.342956+00:00,False,None
4,12875,2,operating,,[193468],2021.0,None,100000318339,2025-08-06 21:45:42.525836+00:00,2026-02-27 12:36:38.709497+00:00,False,None


In [ ]:
# filter status_timeline for only entries in goget_project
status_timeline_check = get_sql_table("status_timeline")

# Convert any timezone-aware datetime columns to timezone-unaware
for col in ['created', 'modified']:
    if col in status_timeline_check.columns and pd.api.types.is_datetime64_any_dtype(status_timeline_check[col]):
        if status_timeline_check[col].dt.tz is not None:
            status_timeline_check[col] = status_timeline_check[col].dt.tz_localize(None)

status_timeline_check.to_excel(export_qc_path + 'status_timeline_check_(all)' + timestamp + '.xlsx', index=False)

In [ ]:
# show all notes with "War Impacted" in them
status_timeline_merged[status_timeline_merged['notes'].str.contains('War Impacted', na=False)]

,id,order,status,substatus,statusDatasource,year,monthOrHalfYear,unit_id,created,modified,delayed,notes
12436,39443,6,mothballed,stated,"[231874, 233183]",2025.0,June,100000312253,2026-03-20 20:25:22.952077+00:00,2026-04-06 13:44:44.288709+00:00,False,War Impacted - Israel launched airstrikes targ...


for some reason, status timelines are not showing when we know there are entries.  [L100001055037](https://gem-project-db.herokuapp.com/projects/edit/100001055037), for example at a discovered year, a start up year, and a current status. but when i check the status timeline for that unit id, it is not there - see row below. It does exist in the goget_project table, but not the status:

project_id = 100001055037
goget_project[goget_project['project_id'] == project_id] (exists!)
status_timeline[status_timeline['unit_id'] == project_id] (blank)




For some reason, country is not in the GOGET tables. as a work around, download all goget fields from the interface and put that in the qc folder. then merge the Country/Area onto goget projct on Unit ID

In [ ]:
# merge namePrefix from goget_project onto reserves_production using project_id
reserves_production = get_sql_table("reserves_production")
reserves_production['project_id'] = reserves_production['project_id'].astype(str)
# Convert goget_project['project_id'] to string type to match reserves_production
goget_project['project_id'] = goget_project['project_id'].astype(str)
reserves_production_merged = reserves_production.merge(goget_project[["project_id", "namePrefix","Country/Area" ]], on="project_id", how="left")

display(reserves_production_merged.head())

,id,productionReserves,fuelDesc,reserveClassification,quantity,quantityUnit,year,reserveProductionDatasource,notes,project_id,namePrefix,Country/Area
0,72057,production,oil and condensate,None,0.40,million bbl/y,2024.0,[223731],None,100000321628,FordWest (Wolfcamp) - Bpx Operating,United States
1,33762,production,gas,None,30207834.00,thousand cubic feet/y,2022.0,[193608],None,100000316204,Great White (AC857) - Shell,United States
2,33763,production,gas,None,5858999.00,thousand cubic feet/y,2022.0,[193608],None,100000316505,Saint Malo (WR678) - Union,United States
3,33,production,gas,None,1.34,million cubic feet/d,2019.0,[193224],None,100000302857,Caricare,Colombia
4,57713,production,oil,None,0.00,million bbl/y,2023.0,[193806],None,100000321637,FordWest (Wolfcamp) - Permian Resources Operating,United States


In [ ]:
# count "makeWIKI" entries
make_wiki_count = goget_project['makeWIKI'].value_counts()

print(make_wiki_count)

makeWIKI
False    9038
True     8187
Name: count, dtype: int64


QC Steps - Production and reserves


In [ ]:
# check if any productionReserves = "production" and "quantityUnit" does not include "/d" "/y" "day" "pd" or "[per year]"
condition = (reserves_production_merged['productionReserves'] == 'production') & \
            (~reserves_production_merged['quantityUnit'].str.contains('/d|/y|day|pd|[per year]', na=False))

filtered_df = reserves_production_merged[condition]

if not filtered_df.empty:
    print("There are entries where productionReserves is 'production' and quantityUnit does not include the specified substrings.")
    display(filtered_df)
else:
    print("No entries found matching the criteria.")

No entries found matching the criteria.


In [ ]:
# check if any productionReserces = reserves and reserveClassification is missing (None, NaN, or empty string)
condition = (reserves_production_merged['productionReserves'] == 'reserves') & \
            (reserves_production_merged['reserveClassification'].isna() | (reserves_production_merged['reserveClassification'] == ''))

filtered_df = reserves_production_merged[condition]

if not filtered_df.empty:
    print("There are entries where productionReserves is 'reserves' and reserveClassification is missing (NaN or empty string).")
    display(filtered_df)
else:
    print("No entries found matching the criteria.")

# 2025 10 23 - kara arna has a reserve type that needs to be added to the db, so its blank rn
# 2026 01 22 - 100001064253	Konta fixed, Kara-Anna sstill needing reserve type
# fixed kara anna

No entries found matching the criteria.


In [ ]:
# check for exact duplicates in production and reserves
def check_exact_duplicates(df):
    # Group by the specified columns and count occurrences
    duplicates = df.groupby(['productionReserves', 'fuelDesc', 'reserveClassification',	'quantity',	'quantityUnit',	'year', 'project_id']).size().reset_index(name='counts')

    # Filter for counts greater than 1
    duplicates = duplicates[duplicates['counts'] > 1]

    return duplicates

duplicates = check_exact_duplicates(reserves_production_merged)

if not duplicates.empty:
  print("Full entry duplicate entries found, see file duplicate_full_entries_" + timestamp + " in QC dump folder")
  # save out to dump
  duplicates.to_csv(export_qc_path + f"duplicate_full_entries_{timestamp}.csv")
else:
  print("No duplicate entries found.")



No duplicate entries found.


In [ ]:
# pull recent production in, filtering by all and specific fuel

# Filter for rows where productionReserves is 'production' or 'production design capacity'
production_df = reserves_production_merged[reserves_production_merged['productionReserves'].isin(['production', 'production design capacity'])].copy() #added pdc of 2026 01 23

# Sort by project_id, fuelDesc, and year in descending order to get the most recent year first
production_df = production_df.sort_values(by=['project_id', 'fuelDesc', 'year'], ascending=[True, True, False])

# Drop duplicate project_id and fuelDesc combinations, keeping the first row (which is the most recent year)
recent_production_byfuel_df = production_df.drop_duplicates(subset=['project_id', 'fuelDesc'])

# Drop duplicates so any fuel counts - use this for recent status
recent_production_overall_df = recent_production_byfuel_df.drop_duplicates(subset=['project_id'])

# rename year to prodyear
recent_production_byfuel_df = recent_production_byfuel_df.rename(columns={'year': 'prodyear'})
recent_production_overall_df = recent_production_overall_df.rename(columns={'year': 'prodyear'})

# Select the desired columns
recent_production_byfuel_df = recent_production_byfuel_df[['project_id', 'namePrefix', 'fuelDesc', 'prodyear', 'quantity', 'quantityUnit','Country/Area']]
recent_production_overall_df = recent_production_overall_df[['project_id', 'namePrefix', 'prodyear', 'quantity', 'quantityUnit','Country/Area']]

display(recent_production_overall_df.head())

,project_id,namePrefix,prodyear,quantity,quantityUnit,Country/Area
38401,100000300002,Abalone,2024.0,0.02,million m³/d,Brazil
54352,100000300004,Acauã,2024.0,8.99,bbl/d,Brazil
49543,100000300006,Aguilhada,2024.0,0.00,million m³/d,Brazil
70433,100000300008,Albacora,2024.0,0.03,million m³/d,Brazil
13623,100000300009,Albacora Leste,2024.0,0.47,million m³/d,Brazil


In [ ]:
#Convert to common units (million bbl and million cubic meters)

# combine 'fuelDesc' and 'quantityUnit' as "Fuel_quantity" in recent_production_byfuel_df
recent_production_byfuel_df['Fuel_quantity'] = recent_production_byfuel_df['fuelDesc'] + ' ' + recent_production_byfuel_df['quantityUnit']

# unique fuel_quanity to make sure all are on conversion sheet
unique_fuel_quantity = recent_production_byfuel_df['Fuel_quantity'].unique()

# import conversion factors
conversion_factors = pd.read_csv('/content/drive/MyDrive/GOGET q4 2025/GOGET oil & gas extraction sites conversion factors for QC  2025 11 18 - Conversion factors.csv')

# compare conversion factors "Fuel description + Units (original)" to unique_fuel_quantity looking for missing entries
missing_entries = []
for entry in unique_fuel_quantity:
    if entry not in conversion_factors['Fuel description + Units (original)'].values:
        missing_entries.append(entry)
        print(f"Missing entry: {entry}") # Print missing entries as they are found

# After the loop, check if any entries were missing
if not missing_entries:
    print("No missing entries found.")

Missing entry: Non-associated gas million cubic feet/d
Missing entry: crude oil bbl/d
Missing entry: coal bed methane million Sm³/y
Missing entry: associated gas million cubic meters/y
Missing entry: coal seam gas tj/d
Missing entry: oil and condensate million bbl/y
Missing entry: sales gas thousand cubic feet/y
Missing entry: crude oil million tonnes/y


In [ ]:
# Apply conversion factors and add new columns

# Ensure conversion_factors has unique 'Fuel description + Units (original)' before merging
# This assumes that for duplicate 'Fuel description + Units (original)', the 'Conversion factor' and 'Common units' are identical or we want to pick one arbitrarily.
# Given the earlier output, they appear to be identical.
unique_conversion_factors = conversion_factors.drop_duplicates(subset=['Fuel description + Units (original)'], keep='first')

recent_production_byfuel_converted_df = recent_production_byfuel_df.merge(
    unique_conversion_factors[['Fuel description + Units (original)', 'Conversion factor', 'Common units']],
    left_on='Fuel_quantity',
    right_on='Fuel description + Units (original)',
    how='left'
)

# Apply the conversion factor to the quantity
recent_production_byfuel_converted_df['quantity_converted'] = recent_production_byfuel_converted_df['quantity'] * recent_production_byfuel_converted_df['Conversion factor']

# Select and rename the desired columns
recent_production_byfuel_converted_df = recent_production_byfuel_converted_df[[
    'project_id',
    'namePrefix',
    'Country/Area',
    'fuelDesc',
    'prodyear',
    'quantity',
    'quantityUnit',
    'quantity_converted',
    'Common units'
]]

In [ ]:
# when "Common units" = "million m³/y" multiply by .00588 new "shared unit" column
recent_production_byfuel_converted_df.loc[recent_production_byfuel_converted_df['Common units'] == 'million m³/y', 'shared unit (million boe/y)'] = recent_production_byfuel_converted_df['quantity_converted'] * .00588

# when common units = million bbl/y or million boe/y, put that value in shared unit column
recent_production_byfuel_converted_df.loc[recent_production_byfuel_converted_df['Common units'].isin(['million bbl/y', 'shared unit (million boe/y)']), 'shared unit (million boe/y)'] = recent_production_byfuel_converted_df['quantity_converted']

# sort recent_production_byfuel_converted_df by shared unit (million boe/y) in ascending order
recent_production_byfuel_converted_df = recent_production_byfuel_converted_df.sort_values(by=['shared unit (million boe/y)'], ascending=[False])


In [ ]:
# sum by country, keeping million bbl/y	and million m³/y seperate, dropping other columns, average year prodyear
country_recent_prod = recent_production_byfuel_converted_df.groupby(['Country/Area', 'fuelDesc'], as_index=False).agg({
    'shared unit (million boe/y)': 'sum'
})

# sort on shared unit (million boe/y) high to low
country_recent_prod = country_recent_prod.sort_values(by=['shared unit (million boe/y)'], ascending=[False])

country_recent_prod

,Country/Area,fuelDesc,shared unit (million boe/y)
325,United States,gas,8326.685338
258,Russia,gas,6347.981791
273,Saudi Arabia,oil,5246.875000
328,United States,oil,4764.759514
262,Russia,oil,3938.145689
...,...,...,...
28,Australia,hydrocarbons,0.000000
319,United Kingdom,hydrocarbons,0.000000
309,United Arab Emirates,hydrocarbons,0.000000
321,United Kingdom,oil and gas,0.000000


In [ ]:
country_recent_prod.to_csv(export_qc_path + 'country_recent_prod_' + timestamp + '.csv', index=False)

In [ ]:
# follow the same process for reserves
# pull recent production in, filtering by all and specific fuel

# Filter for rows where productionReserves is 'reserves"
reserves_df = reserves_production_merged[reserves_production_merged['productionReserves'].isin(['reserves'])].copy()

# Sort by project_id, fuelDesc, reserveClassification, and year in descending order to get the most recent year first
reserves_df = reserves_df.sort_values(by=['project_id', 'fuelDesc', 'reserveClassification', 'year'], ascending=[True, True, True, False])

# Drop duplicate project_id, fuelDesc, and reserveClassification combinations, keeping the first row (which is the most recent year for that combination)
recent_reserves_byfuel_df = reserves_df.drop_duplicates(subset=['project_id', 'fuelDesc', 'reserveClassification'])

# Drop duplicates so only one reserve entry is kept for each project (the most recent one based on the previous sort)
recent_reserves_overall_df = recent_reserves_byfuel_df.drop_duplicates(subset=['project_id'])

# rename year to prodyear
recent_reserves_byfuel_df = recent_reserves_byfuel_df.rename(columns={'year': 'reserveyear'})
recent_reserves_overall_df = recent_reserves_overall_df.rename(columns={'year': 'reserveyear'})

# Select the desired columns
recent_reserves_byfuel_df = recent_reserves_byfuel_df[['project_id', 'namePrefix', 'fuelDesc', 'reserveyear', 'reserveClassification', 'quantity', 'quantityUnit','Country/Area']]
recent_reserves_overall_df = recent_reserves_overall_df[['project_id', 'namePrefix', 'reserveyear', 'reserveClassification', 'quantity', 'quantityUnit','Country/Area']]

display(recent_reserves_overall_df.head())

,project_id,namePrefix,reserveyear,reserveClassification,quantity,quantityUnit,Country/Area
9451,100000300001,Matzen,2014.0,initial hydrocarbons in place,289.00,million boe,Austria
28532,100000300002,Abalone,2015.0,original volume in place,3137.00,million m³,Brazil
26881,100000300003,Acajá-Burizinho,2014.0,volume in place,1.98,million m³,Brazil
32087,100000300004,Acauã,2015.0,volume in place,3.14,million m³,Brazil
62345,100000300006,Aguilhada,2015.0,volume in place,62.89,million m³,Brazil


In [ ]:
#Convert to common units (million bbl and million cubic meters)

# combine 'fuelDesc' and 'quantityUnit' as "Fuel_quantity" in recent_reserves_byfuel_df
recent_reserves_byfuel_df['Fuel_quantity'] = recent_reserves_byfuel_df['fuelDesc'] + ' ' + recent_reserves_byfuel_df['quantityUnit']

# unique fuel_quanity to make sure all are on conversion sheet
unique_fuel_quantity = recent_reserves_byfuel_df['Fuel_quantity'].unique()

# import conversion factors
conversion_factors = pd.read_csv('/content/drive/MyDrive/GOGET q4 2025/GOGET oil & gas extraction sites conversion factors for QC  2025 11 18 - Conversion factors.csv')


# compare conversion factors "Fuel description + Units (original)" to unique_fuel_quantity looking for missing entries
missing_entries = []
for entry in unique_fuel_quantity:
    if entry not in conversion_factors['Fuel description + Units (original)'].values:
        missing_entries.append(entry)
        print(f"Missing entry: {entry}") # Print missing entries as they are found
# After the loop, check if any entries were missing
if not missing_entries:
    print("No missing entries found.")


Missing entry: oil million stock tank bbls
Missing entry: gas condensate tonnes
Missing entry: associated gas bcm


In [ ]:
# Apply conversion factors and add new columns

# Ensure conversion_factors has unique 'Fuel description + Units (original)' before merging
# This assumes that for duplicate 'Fuel description + Units (original)', the 'Conversion factor' and 'Common units' are identical or we want to pick one arbitrarily.
# Given the earlier output, they appear to be identical.
unique_conversion_factors = conversion_factors.drop_duplicates(subset=['Fuel description + Units (original)'], keep='first')

recent_reserves_byfuel_converted_df = recent_reserves_byfuel_df.merge(
    unique_conversion_factors[['Fuel description + Units (original)', 'Conversion factor', 'Common units']],
    left_on='Fuel_quantity',
    right_on='Fuel description + Units (original)',
    how='left'
)

# Apply the conversion factor to the quantity
recent_reserves_byfuel_converted_df['quantity_converted'] = recent_reserves_byfuel_converted_df['quantity'] * recent_reserves_byfuel_converted_df['Conversion factor']

# Select and rename the desired columns
recent_reserves_byfuel_converted_df = recent_reserves_byfuel_converted_df[[
    'project_id',
    'namePrefix',
    'Country/Area',
    'fuelDesc',
    'reserveClassification',
    'reserveyear',
    'quantity',
    'quantityUnit',
    'quantity_converted',
    'Common units'
]]

In [ ]:
# when "Common units" = "million m³/y" multiply by .00588 new "shared unit" column
recent_reserves_byfuel_converted_df.loc[recent_reserves_byfuel_converted_df['Common units'] == 'million m³', 'shared unit (million boe)'] = recent_production_byfuel_converted_df['quantity_converted'] * .00588

# when common units = million bbl/y or million boe/y, put that value in shared unit column
recent_reserves_byfuel_converted_df.loc[recent_reserves_byfuel_converted_df['Common units'].isin(['million bbl', 'shared unit (million boe)']), 'shared unit (million boe)'] = recent_reserves_byfuel_converted_df['quantity_converted']

# sort recent_production_byfuel_converted_df by shared unit (million boe/y) in ascending order
recent_reserves_byfuel_converted_df = recent_reserves_byfuel_converted_df.sort_values(by=['shared unit (million boe)'], ascending=[False])


In [ ]:
# Select relevant columns for production and rename for clarity
prod_compare_df = recent_production_byfuel_converted_df[['project_id', 'shared unit (million boe/y)']].rename(columns={'shared unit (million boe/y)': 'production_boe'})

# Select relevant columns for reserves and rename for clarity
res_compare_df = recent_reserves_byfuel_converted_df[['project_id', 'shared unit (million boe)']].rename(columns={'shared unit (million boe)': 'reserves_boe'})

# sum all values for each project id in both dfs
prod_compare_df = prod_compare_df.groupby('project_id').sum().reset_index()
res_compare_df = res_compare_df.groupby('project_id').sum().reset_index()

# Merge the two dataframes on project_id
# Using an inner merge to only include projects present in both production and reserves
comparison_df = pd.merge(prod_compare_df, res_compare_df, on='project_id', how='inner')

# Filter for entries where production value is higher than reserves value
projects_production_gt_reserves = comparison_df[comparison_df['production_boe'] > comparison_df['reserves_boe']]

# caculate the difference between production and reserves
projects_production_gt_reserves['difference'] = projects_production_gt_reserves['production_boe'] - projects_production_gt_reserves['reserves_boe']

# sort on difference
projects_production_gt_reserves = projects_production_gt_reserves.sort_values(by=['difference'], ascending=[False])

# Display the results
if not projects_production_gt_reserves.empty:
    print("Projects where production value is higher than reserves value:")
    display(projects_production_gt_reserves)
else:
    print("No projects found where production value is higher than reserves value.")

Projects where production value is higher than reserves value:


/tmp/ipykernel_11068/3126312836.py:19: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  projects_production_gt_reserves['difference'] = projects_production_gt_reserves['production_boe'] - projects_production_gt_reserves['reserves_boe']


,project_id,production_boe,reserves_boe,difference
2884,100000312037,8.232000e+02,2.226046e+02,6.005954e+02
2903,100000312067,3.292800e+02,3.874986e-01,3.288925e+02
4581,100000318296,2.657760e+02,2.041712e+00,2.637343e+02
3319,100000313066,2.574095e+02,8.510658e+00,2.488988e+02
4514,100000317603,2.270766e+02,5.990044e-03,2.270707e+02
...,...,...,...,...
747,100000301647,5.350800e-03,5.330233e-03,2.056695e-05
2271,100000309989,6.927228e-06,0.000000e+00,6.927228e-06
436,100000301133,4.057200e-06,0.000000e+00,4.057200e-06
335,100000300958,1.411200e-06,6.357456e-10,1.410564e-06


In [ ]:
# save projects_production_gt_reserves to qc dump
projects_production_gt_reserves.to_csv(export_qc_path + timestamp +'projects_production_gt_reserves.csv', index=False)

In [ ]:
# save out unique reserveClassification csv for conversion first pass
unique_reserve_classification = recent_reserves_byfuel_converted_df['reserveClassification'].unique()
unique_reserve_classification_df = pd.DataFrame(unique_reserve_classification, columns=['reserveClassification'])
unique_reserve_classification_df.to_csv(f"{export_qc_path}unique_reserve_classification_{timestamp}.csv", index=False)

In [ ]:
# Filter out rows where substatus is 'planned'
filtered_status_timeline = status_timeline_merged[status_timeline_merged['substatus'] != 'planned'].copy()

# sort by order, low to high
filtered_status_timeline = filtered_status_timeline.sort_values(by=['order'])

# Drop duplicate unit_id, keeping the lowest ranked row (which is the most recent status)
current_status_rank_df = filtered_status_timeline.drop_duplicates(subset=['unit_id'], keep='last')

# Rename 'unit_id' to 'project_id' for clarity and select relevant columns
current_status_rank_df = current_status_rank_df.rename(columns={'unit_id': 'project_id'})
current_status_rank_df = current_status_rank_df[['project_id', 'status', 'substatus', 'year']]

display(current_status_rank_df.head())

,project_id,status,substatus,year
26654,100000321239,discovered,early,2023.0
21269,100000320901,discovered,early,2023.0
20524,100000313919,discovered,,2020.0
20326,100000310998,operating,,2015.0
4968,100000312890,discovered,,1986.0


In [ ]:
# current status was caclucated by most recent year, now we are using rank
current_status_df = current_status_rank_df # using rank over year for most recent data

In [ ]:
# create new df with name, most recent production and most recent status
status_production = goget_project.copy()
current_status_df['project_id'] = current_status_df['project_id'].astype(str)
status_production = status_production.merge(current_status_df, on='project_id', how='left')

# Merge with recent_production_overall_df, specifying custom suffixes
# This will result in columns like 'namePrefix_goget' and 'Country/Area_goget' from the left dataframe (status_production)
# and 'namePrefix_prod' and 'Country/Area_prod' from the right dataframe (recent_production_overall_df)
status_production = status_production.merge(
    recent_production_overall_df,
    on='project_id',
    how='left',
    suffixes=('_goget', '_prod')
)

# Select only the desired columns.
# We assume 'namePrefix_goget' corresponds to the original 'namePrefix' from goget_project.
# Other 'Country/Area' columns are implicitly dropped as they are not selected.
status_production = status_production[['project_id', 'namePrefix_goget', 'status', 'substatus', 'year', 'prodyear', 'quantity', 'quantityUnit']]
status_production = status_production.rename(columns={'namePrefix_goget': 'namePrefix'})

display(status_production.head())

,project_id,namePrefix,status,substatus,year,prodyear,quantity,quantityUnit
0,100000313656,Onward N Basal Mannville,operating,,2017.0,2017.0,7520.4,m³/y
1,100000314273,ST037 - Arena,operating,,2023.0,2023.0,1022645.0,thousand cubic feet/y
2,100000314274,EI276 - Arena,operating,,2023.0,2023.0,627296.0,thousand cubic feet/y
3,100000314276,HI545A - Arena,operating,,2023.0,2023.0,2917017.0,thousand cubic feet/y
4,100000314278,EI238 - Arena,operating,,2023.0,2023.0,2425067.0,thousand cubic feet/y


In [ ]:
#filter for where prodyear is greater than 2020,  but status	is blank
filtered_df = status_production[(status_production['prodyear'] > 2020) & (status_production['status'].isna())]

# filter out for when quantity is 0
filtered_df = filtered_df[filtered_df['quantity'] > 0]

In [ ]:
filtered_df # all of these acutally have a status, but my code is missing it. for some reason, they appear in the goget_project, but not in the status_timeline

,project_id,namePrefix,status,substatus,year,prodyear,quantity,quantityUnit
46,100001061418,Aphrodite,NaN,NaN,NaN,2027.0,107.00,million scf/d
66,100001082402,Midway-Sunset - Sentinel Peak Resources Califo...,NaN,NaN,NaN,2024.0,0.04,million bbl/y
114,100001082404,San Ardo - Chevron U.S.A.,NaN,NaN,NaN,2024.0,0.01,million bbl/y
165,100001082493,"Brookeland (Austin Chalk, 8800) - Burk Royalty...",NaN,NaN,NaN,2024.0,196.72,million m³/y
166,100001082495,Slaughter - Occidental Permian Ltd. (Conventi...,NaN,NaN,NaN,2024.0,44.66,million m³/y
...,...,...,...,...,...,...,...,...
17160,100001082145,Fillmore - Crowheart Energy,NaN,NaN,NaN,2024.0,0.01,million bbl/y
17164,100001082112,Crawford Draw - Sage Butte Prb Operating,NaN,NaN,NaN,2024.0,0.01,million bbl/y
17171,100001082164,Green River Bend - Greylock Production,NaN,NaN,NaN,2024.0,0.11,million bbl/y
17186,100001082183,Hogsback South - Urban Oil & Gas Group,NaN,NaN,NaN,2024.0,0.01,million bbl/y


In [ ]:
# this checks if an unit has multiple reserve data points from the same year # as of 2026 01 23 - these are just Kara Ana, which will be fixed once that reserve type is in
def check_duplicates(df):
    # Group by the specified columns and count occurrences
    close_to_duplicates = df.groupby(['project_id', 'productionReserves','reserveClassification' ,'fuelDesc', 'year']).size().reset_index(name='counts')

    # Filter for counts greater than 1
    close_to_duplicates = close_to_duplicates[close_to_duplicates['counts'] > 1]

    return close_to_duplicates

close_to_duplicates = check_duplicates(reserves_production_merged)

if not close_to_duplicates.empty:
  print("Repeat year, fuel description, entries found, see file duplicate_entries_"  +timestamp + " in QC dump folder")
  # save out to dump
  close_to_duplicates.to_csv(export_qc_path + f"duplicate_reserve_entries_{timestamp}.csv")
else:
  print("No duplicate entries found.")

No duplicate entries found.


In [ ]:
# this checks if an unit has multiple production data points from the same year - as of 2025-10-24 there were no entries found here!
def check_prod_duplicates(df):

    # filter for just production entries
    prod_dups_check = df[df['productionReserves'] == 'production']
    # Group by the specified columns and count occurrences
    prod_dups_check = prod_dups_check.groupby(['project_id' , 'namePrefix', 'fuelDesc', 'year']).size().reset_index(name='counts')

    # Filter for counts greater than 1
    prod_dups_check = prod_dups_check[prod_dups_check['counts'] > 1]

    return prod_dups_check

prod_dups_check = check_prod_duplicates(reserves_production_merged)

if not prod_dups_check.empty:
  print("Repeat year, fuel description, entries found see file duplicate_production_"  +timestamp + " in QC dump folder")
  # save out to dump
  prod_dups_check.to_csv(export_qc_path + f"duplicate_production_{timestamp}.csv")
else:
  print("No duplicate entries found.")

Repeat year, fuel description, entries found see file duplicate_production_2026-03-16_21-28-04 in QC dump folder


In [ ]:
# discovery to start up timeline
disco_to_start = status_timeline_merged.copy()

# discovery year = earliest status entry where status = "discovered"
disco_to_start['Discovery Year'] = status_timeline_merged[status_timeline_merged['status'] == 'discovered'].groupby('unit_id')['year'].transform('min')

# start year = earliest status of "operating"
disco_to_start['Start Year'] = status_timeline_merged[(status_timeline_merged['status'] == 'operating') & (status_timeline_merged['substatus'] == 'ramp up')].groupby('unit_id')['year'].transform('min')

In [ ]:
# Extract unique Discovery Year for each unit_id
discovery_years = disco_to_start.dropna(subset=['Discovery Year'])[['unit_id', 'Discovery Year']].drop_duplicates(subset=['unit_id'])

# Extract unique Start Year for each unit_id
start_years = disco_to_start.dropna(subset=['Start Year'])[['unit_id', 'Start Year']].drop_duplicates(subset=['unit_id'])

# Merge the two DataFrames on unit_id to get one row per unit_id
discovery_start_years = pd.merge(discovery_years, start_years, on='unit_id', how='outer')

# calculate the discovery to start year for each unit. If both values are not blank Start Year - Discovery Year
discovery_start_years['Discovery to Start Year'] = discovery_start_years['Start Year'] - discovery_start_years['Discovery Year']

In [ ]:
# sort discovery_start_years on Discovery to Start Year
discovery_start_years = discovery_start_years.sort_values(by=['Discovery to Start Year'])

# define problematic data
problematic_data = discovery_start_years[(discovery_start_years['Discovery to Start Year'] < 0) | (discovery_start_years['Discovery to Start Year'] > 60)]

# display problematic data
display(problematic_data)

,unit_id,Discovery Year,Start Year,Discovery to Start Year
4776,100000311733,1955.0,2016.0,61.0
747,100000300915,1949.0,2010.0,61.0
5558,100000312624,1959.0,2020.0,61.0
8829,100000321055,1956.0,2020.0,64.0
8808,100000320950,1957.0,2025.0,68.0
3393,100000308719,1946.0,2015.0,69.0


In [ ]:
goget_project.columns

Index(['id_x', 'basinDatasource', 'concessionBlockDatasource',
       'projectComplexDatasource', 'offshore', 'offshoreDatasource',
       'productionType', 'productionTypeDatasource', 'fuelType', 'unitType',
       'basin', 'concessionBlock', 'project_id', 'projectComplex_id',
       'makeWIKI', 'namePrefix', 'concessionBlock2', 'concessionBlock3',
       'concessionBlock4', 'concessionBlock5', 'concessionBlock6',
       'concessionBlock7', 'capex', 'plant_table_id_x', 'Country/Area',
       'plant_table_id_y', 'country_id', 'latitude', 'longitude'],
      dtype='object')

In the feb 2024 release, 1998 fields had a start year and discovery year

In [ ]:
# keep just lat long and "Unit ID" column
GOGET_coordinates = goget_project[['project_id', 'latitude', 'longitude']]


# Merge wkt (coordinates) from goget_project_wkt onto status_timeline_merged
# The unit_id in status_timeline_merged corresponds to project_id in goget_project_wkt
status_timeline_merged['unit_id'] = status_timeline_merged['unit_id'].astype(str)
status_timeline_with_coord = status_timeline_merged.merge(
    GOGET_coordinates[['project_id', 'latitude', 'longitude']],
    left_on='unit_id',
    right_on='project_id',
    how='left'
)

# Drop the redundant 'project_id' column that resulted from the merge if you only need 'unit_id'
status_timeline_with_coord = status_timeline_with_coord.drop(columns=['project_id'])

print("First 5 rows of status_timeline_with_wkt with coordinates:")
display(status_timeline_with_coord.head())

First 5 rows of status_timeline_with_wkt with coordinates:


/tmp/ipykernel_11068/2658040894.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  status_timeline_merged['unit_id'] = status_timeline_merged['unit_id'].astype(str)


,id,order,status,substatus,statusDatasource,year,monthOrHalfYear,unit_id,created,modified,delayed,notes,latitude,longitude
0,19876,1,discovered,,[193245],1978.0,None,100000309032,2025-09-10 16:46:04.994321+00:00,2026-02-27 12:43:12.552439+00:00,False,None,54.1100,-110.9670
1,17136,2,operating,ramp up,[193191],1995.0,None,100000301072,2025-09-10 16:45:45.326477+00:00,2026-02-27 12:30:06.169041+00:00,False,None,53.4490,3.6590
2,24670,3,operating,,[212075],2025.0,,100000312250,2025-09-10 16:46:37.662886+00:00,2026-02-27 12:43:43.871888+00:00,False,,26.8614,52.1813
3,8352,1,discovered,,[193400],1963.0,None,100000302911,2025-08-06 21:45:09.234914+00:00,2026-02-27 12:44:28.342956+00:00,False,None,7.2640,-73.5130
4,12875,2,operating,,[193468],2021.0,None,100000318339,2025-08-06 21:45:42.525836+00:00,2026-02-27 12:36:38.709497+00:00,False,None,13.0955,-59.4759


In [ ]:
# check for 2025 FIDs
fid_2025 = status_timeline_merged[(status_timeline_merged['status'] == 'in-development') & (status_timeline_merged['substatus'] == 'actual') & (status_timeline_merged['year'] == 2025)]

display(fid_2025)

,id,order,status,substatus,statusDatasource,year,monthOrHalfYear,unit_id,created,modified,delayed,notes
91,39337,0,in-development,actual,[200688],2025.0,June,100000312281,2026-02-11 22:32:59.655631+00:00,2026-02-27 12:43:38.929535+00:00,False,
19449,34693,7,in-development,actual,[208333],2025.0,September,100000319556,2025-09-30 15:34:07.206226+00:00,2026-02-27 12:37:52.438727+00:00,False,
19464,11405,2,in-development,actual,"[186272, 190876, 191886, 216831]",2025.0,January,100000313101,2025-08-06 21:45:31.639581+00:00,2026-02-27 12:32:44.474114+00:00,False,
19484,35653,7,in-development,actual,[219602],2025.0,November,100000317090,2025-12-04 17:18:19.128697+00:00,2026-02-27 12:33:55.501467+00:00,False,Malaysian energy giant Petronas and its partne...
19825,12975,4,in-development,actual,"[187081, 190803]",2025.0,March,100000318408,2025-08-06 21:45:43.163429+00:00,2026-02-27 12:32:47.173196+00:00,False,"""bp Trinidad and Tobago has achieved two major..."
20055,35654,7,in-development,actual,[219602],2025.0,November,100000317089,2025-12-04 17:19:10.578330+00:00,2026-02-27 12:28:11.132449+00:00,False,
20297,24470,2,in-development,actual,"[196796, 220734]",2025.0,March,100000312173,2025-09-10 16:46:36.207750+00:00,2026-02-27 12:43:02.212042+00:00,False,"""Shell Brasil Petróleo Ltda. (Shell Brasil), a..."
20410,11379,4,in-development,actual,[207353],2025.0,September,100000313089,2025-08-06 21:45:31.419043+00:00,2026-02-27 12:43:30.811201+00:00,False,""" ExxonMobil has made a final investment decis..."
20850,35652,8,in-development,actual,[219602],2025.0,November,100000314096,2025-12-04 17:15:58.688822+00:00,2026-02-27 12:43:46.722034+00:00,False,FID
21995,34707,5,in-development,actual,[208588],2025.0,February,100000321181,2025-10-01 18:39:38.659749+00:00,2026-02-27 12:38:16.686873+00:00,False,


In [ ]:
# check for 2025 discoveries
disco_2025 = status_timeline_merged[(status_timeline_merged['status'] == 'discovered') & (status_timeline_merged['substatus'] == 'early') & (status_timeline_merged['year'] == 2025)]

display(disco_2025)

,id,order,status,substatus,statusDatasource,year,monthOrHalfYear,unit_id,created,modified,delayed,notes
8709,14429,1,discovered,early,"[187081, 211171]",2025.0,March,100000323354,2025-08-06 21:45:53.211082+00:00,2026-02-27 12:32:47.242232+00:00,False,
21706,35124,8,discovered,early,[212773],2025.0,April,100000318397,2025-10-23 21:27:39.137990+00:00,2026-02-27 12:32:45.406042+00:00,False,"Delayed, the US revoked a licenses it gave to ..."
21811,35263,4,discovered,early,[214237],2025.0,October,100000323385,2025-10-31 21:51:43.448796+00:00,2026-02-27 12:45:30.544773+00:00,False,"Elevated Co2, but BP says its still commercial"
22307,35368,6,discovered,early,[205912],2025.0,,100000319561,2025-11-12 17:40:14.570143+00:00,2026-02-27 12:28:50.579894+00:00,False,
23869,34258,1,discovered,early,"[202928, 200700]",2025.0,,100000323343,2025-09-10 16:47:48.280455+00:00,2026-02-27 12:34:03.482930+00:00,False,
23887,34342,1,discovered,early,"[222472, 199354]",2025.0,August,100000323483,2025-09-10 16:47:48.850225+00:00,2026-02-27 12:34:32.808537+00:00,False,
23898,34343,1,discovered,early,"[198840, 222472]",2025.0,April,100000323484,2025-09-10 16:47:48.856999+00:00,2026-02-27 12:34:17.832032+00:00,False,
26828,34748,5,discovered,early,[209142],2025.0,August,100000321240,2025-10-03 18:44:56.128117+00:00,2026-02-27 12:34:57.252948+00:00,False,
26963,34749,5,discovered,early,[209152],2025.0,July,100000321241,2025-10-03 18:53:47.959079+00:00,2026-02-27 12:27:36.686000+00:00,False,
28333,34296,1,discovered,early,[199752],2025.0,August,100000323385,2025-09-10 16:47:48.539787+00:00,2026-02-27 12:45:30.542861+00:00,False,


In [ ]:
# Function to check for start-ups in a given year
def count_startups_by_year(year, status_timeline_df):
    startups = status_timeline_df[
        (status_timeline_df['status'] == 'operating') &
        (status_timeline_df['substatus'] == 'ramp up') &
        (status_timeline_df['year'] == year)
    ]
    return startups.shape[0]

# Years to check
years_to_check = [2025, 2024, 2023, 2022, 2021, 2020]

# Display counts for each year
for year in years_to_check:
    count = count_startups_by_year(year, status_timeline_merged)
    print(f"{year} start-ups: {count}")

2025 start-ups: 46
2024 start-ups: 64
2023 start-ups: 75
2022 start-ups: 69
2021 start-ups: 81
2020 start-ups: 67


In [ ]:
# make a df for each of the years in years to check then exports as a tabbed excel with each year having its own tab

# Create an empty dictionary to store DataFrames for each year
year_startups_dfs = {}

for year in years_to_check:
    # Filter status_timeline_merged for startups in the current year
    startups_for_year = status_timeline_with_coord[
        (status_timeline_with_coord['status'] == 'operating') &
        (status_timeline_with_coord['substatus'] == 'ramp up') &
        (status_timeline_with_coord['year'] == year)
    ].copy()

    # Convert any timezone-aware datetime columns to timezone-unaware
    for col in ['created', 'modified']:
        if col in startups_for_year.columns and pd.api.types.is_datetime64_any_dtype(startups_for_year[col]):
            if startups_for_year[col].dt.tz is not None:
                startups_for_year[col] = startups_for_year[col].dt.tz_localize(None)

    # Ensure project_id is string type for merging
    startups_for_year['unit_id'] = startups_for_year['unit_id'].astype(str)
    goget_project['project_id'] = goget_project['project_id'].astype(str)

    # Merge with goget_project to get namePrefix
    startups_for_year_merged = startups_for_year.merge(
        goget_project[['project_id', 'namePrefix']],
        left_on='unit_id',
        right_on='project_id',
        how='left'
    )

    # Store the DataFrame in the dictionary
    year_startups_dfs[f'Startups_{year}'] = startups_for_year_merged

# Define the output Excel file path
excel_output_path = export_qc_path + f'yearly_startups_{timestamp}.xlsx'

# Create a Pandas Excel writer using XlsxWriter as the engine.
# This allows writing multiple sheets to the same file.
with pd.ExcelWriter(excel_output_path, engine='xlsxwriter') as writer:
    for sheet_name, df in year_startups_dfs.items():
        df.to_excel(writer, sheet_name=sheet_name, index=False)

print(f"Yearly startups exported to: {excel_output_path}")

Yearly startups exported to: /content/drive/Shareddrives/GEM Shared Drive/Programs/Oil & Gas Program/Global Oil and Gas Extraction Tracker (GOGET)/QC/QC Output Dump/yearly_startups_2026-03-16_21-28-04.xlsx


In [ ]:
# merge name for goget_project to disco_2025 on unit_id
disco_2025 = disco_2025.merge(goget_project[['project_id', 'namePrefix']], left_on='unit_id', right_on='project_id', how='left')

# merge name from goget_project to fid_2025 on unit_id
fid_2025 = fid_2025.merge(goget_project[['project_id', 'namePrefix']], left_on='unit_id', right_on='project_id', how='left')

# merge namme from goget_project to sstart_2025 on unit id
start_2025 = start_2025.merge(goget_project[['project_id', 'namePrefix']], left_on='unit_id', right_on='project_id', how='left')

NameError: name 'start_2025' is not defined

In [ ]:
# save recent_production_df to export_qc_path
status_production.to_csv(export_qc_path + 'status_production_' + timestamp + '.csv', index=False)
filtered_df.to_csv(export_qc_path + 'filtered_df_' + timestamp + '.csv', index=False)
current_status_df.to_csv(export_qc_path + 'current_status_df_' + timestamp + '.csv', index=False)
recent_production_byfuel_df.to_csv(export_qc_path + 'recent_production_byfuel_df_' + timestamp + '.csv', index=False)
recent_production_byfuel_converted_df.to_csv(export_qc_path + 'recent_production_byfuel_converted_df_' + timestamp + '.csv', index=False)
fid_2025.to_csv(export_qc_path + 'fid_2025_' + timestamp + '.csv', index=False)
disco_2025.to_csv(export_qc_path + 'disco_2025_' + timestamp + '.csv', index=False)
# start_2025.to_csv(export_qc_path + 'start_2025_' + timestamp + '.csv', index=False)